# Overview of the Action-Conditioned (AC) Predictor

![ac_predictor_overview](../resources/v1_0/ac_predictor_overview.png)

The BioJEPA-AC model is designed to take a cell state and a perturbation and predict a latent representation of the perturbed cell state. We use the [Cell State Encoder](https://github.com/GPTomics/biojepa/blob/main/layer_explainers/explainer_cell_state_encoder_v1_0.ipynb) to take the input cell state and generate a representation. We also use the [Action Composer](https://github.com/GPTomics/biojepa/blob/main/layer_explainers/explainer_action_composer_v1_0.ipynb) to generate a unified perturbation latent. The ACPredictor is a transformer-based module that takes these latents and generates a predicted representation of the perturbed cell state, including an uncertainty estimate.

This notebook will walk through each layer so that the reader can build an intuition for what each layer is doing to the data. To this end, you'll see that we set the layer initializations and values to ones so you can calculate them by hand if you need to follow a layer more closely.

In [1]:
import torch
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import math

## Data Prep

We'll start with a simple data prep. The ACPredictor takes three inputs: the cell state latent (output by the Cell State Encoder), target indices to predict, and the perturbation latent (output by the Action Composer). We'll mock all three using small dimensions that are easy to trace by hand.

The context latent and predicted target latent live in the same Cell State Encoder space, so they use the same dimension. The predictor does not need to do all of its work at that dimension, though. We'll first project the context latent down into a smaller predictor embedding, run the transformer layers in that space, and then project the predictions back to the Cell State Encoder dimension. For our example, we'll go from 6 context dimensions to 4 predictor dimensions and then back to 6 output dimensions.

*We'll use dimensions that match our other explainers but generate simpler data to make it easier to keep track of.*

In [2]:
batch = 2
num_genes = 8
context_dim = 6
embed_dim = 4
output_dim = context_dim
n_perts = 2
action_dim = 5
heads = 2
mlp_ratio = 4.0


SEED = 1337
torch.manual_seed(SEED)
np.random.seed(SEED)

In [3]:
def mock_data(dim_1, dim_2, low=1, high=9):
    return torch.from_numpy(np.round(np.random.uniform(low, high, size=(dim_1, dim_2)), 0)).float()

**Cell State Latent (context_latents)**

This is the representation of the control cell as output by the Cell State Encoder, representing the gene latent embeddings for the non-perturbed cell. As a reminder, this shape is $[B, \text{num\_genes}, \text{context\_dim}]$.

In [4]:
context_latents = mock_data(batch * num_genes, context_dim).reshape(batch, num_genes, context_dim)
context_latents.shape, context_latents

(torch.Size([2, 8, 6]),
 tensor([[[3., 2., 3., 5., 4., 5.],
          [3., 9., 7., 2., 4., 6.],
          [2., 9., 5., 7., 7., 4.],
          [4., 6., 7., 3., 3., 6.],
          [5., 2., 4., 3., 5., 8.],
          [2., 9., 4., 5., 5., 1.],
          [7., 9., 2., 2., 4., 7.],
          [1., 4., 1., 7., 4., 7.]],
 
         [[5., 9., 8., 6., 6., 9.],
          [3., 1., 2., 9., 9., 5.],
          [4., 7., 1., 7., 6., 3.],
          [6., 8., 2., 8., 4., 7.],
          [3., 5., 2., 5., 5., 4.],
          [7., 5., 7., 7., 4., 8.],
          [7., 8., 4., 5., 8., 6.],
          [5., 3., 9., 5., 7., 3.]]]))

**Target Indices (target_indices)**

We've built our model so that each cell does not have to have every gene predicted. What is predicted is controlled via target indices. The predictor uses a learned embedding per gene to create query tokens in the attention to drive prediction only on what's desired. Currently, though, our code is just predicting all genes, so our example will follow suit.

In [5]:
target_indices = torch.arange(num_genes).unsqueeze(0).expand(batch, -1)
target_indices.shape, target_indices

(torch.Size([2, 8]),
 tensor([[0, 1, 2, 3, 4, 5, 6, 7],
         [0, 1, 2, 3, 4, 5, 6, 7]]))

**Perturbation Embedding (action_latents)**

This is the representation of the perturbations for the cell as output by the Action Composer. As a reminder, this shape is $[B, N_\text{pert}, \text{action\_dim}]$.

In [6]:
action_latents = mock_data(batch * n_perts, action_dim).reshape(batch, n_perts, action_dim)
action_latents.shape, action_latents

(torch.Size([2, 2, 5]),
 tensor([[[4., 4., 5., 2., 6.],
          [1., 3., 7., 8., 6.]],
 
         [[4., 8., 9., 3., 4.],
          [7., 4., 1., 4., 5.]]]))

## Forward Pass

The goal of this model is to move the context_latent based on the action_latent to a new position in the latent representation of cell states. We do this by:

1. Projecting the context and action latents into the predictor embedding dimension.
2. Creating target gene query embeddings and concatenating them with the context.
3. Using cross-attention to inject the perturbation into the gene representations.
4. Using self-attention and nonlinearity to update the combined representation.
5. Projecting the mean and log-variance predictions back into the Cell State Encoder dimension.

We'll start by projecting the action latent into the predictor embedding dimensions using an MLP-like layer.

### Adapter Projection MLP

The action composer's output lives in a different dimensional space than the predictor embedding, so we need to bridge the gap to give the model a chance to identify how the perturbation impacts the different channels. Instead of a simple linear projection, we'll use a multilayer perceptron with two layers. The nonlinearity between the two layers allows the adapter to learn a nonlinear mapping between the two spaces.

#### Adapter - Linear Layer 1

The first linear layer does the projection from the action dimensions to the embedding dimensions. We'll do incremental weights so you'll see the impact of each upscaling.

In [7]:
a_l1 = nn.Linear(action_dim, embed_dim)
vs, d = action_dim, embed_dim
rows = torch.full((vs,), 0.1).unsqueeze(0)
cols = torch.arange(d).unsqueeze(1)
pattern = 1 * (rows + 0.1 * cols)
a_l1.weight = nn.Parameter(pattern)
nn.init.zeros_(a_l1.bias)
a_l1.weight, a_l1.bias

(Parameter containing:
 tensor([[0.1000, 0.1000, 0.1000, 0.1000, 0.1000],
         [0.2000, 0.2000, 0.2000, 0.2000, 0.2000],
         [0.3000, 0.3000, 0.3000, 0.3000, 0.3000],
         [0.4000, 0.4000, 0.4000, 0.4000, 0.4000]], requires_grad=True),
 Parameter containing:
 tensor([0., 0., 0., 0.], requires_grad=True))

In [8]:
action_emb = a_l1(action_latents)
action_emb.shape, action_emb

(torch.Size([2, 2, 4]),
 tensor([[[ 2.1000,  4.2000,  6.3000,  8.4000],
          [ 2.5000,  5.0000,  7.5000, 10.0000]],
 
         [[ 2.8000,  5.6000,  8.4000, 11.2000],
          [ 2.1000,  4.2000,  6.3000,  8.4000]]], grad_fn=<ViewBackward0>))

#### RMSNorm

Our next layer is normalization. For our modern transformer, we use root mean square normalization, or RMSNorm. RMSNorm calculates the following:
$$
y = \frac{x}{\sqrt{\frac{1}{n} \sum_{i=1}^{n} x_i^2 + \epsilon}} \cdot \gamma
$$

The main reason we use RMSNorm is that it executes faster and uses less memory than standard layer normalization. This efficiency is achieved by entirely removing the mean-centering calculation, which reduces the total number of arithmetic operations and hardware synchronization steps. Recall that normalization is primarily used for large-scale training stability (preventing gradient explosion/vanishing). For deep architectures like ours, the mean of the pre-activation inputs naturally stays close to zero during training, so removing the mean-centering operation preserves the critical variance-bounding effect. Also, the model learns to absorb any minor activation shifts into the subsequent linear weights or the learned affine parameters. Because of this, we're able to use a more efficient normalization.

Since we reuse RMSNorm many times, we'll create a class for it. In the class, you can see that we split out the calculation: first we promote the input to float32 so that the square and reciprocal square root are calculated more reliably, then we normalize by the root mean square and apply the learned per-channel weights. Finally, we cast the result back to the input's original data type.

Since our linear layer produced values with an incremental pattern across channels, you'll see that RMSNorm will rescale them so that the RMS magnitude becomes 1 while preserving the relative differences between channels.

In [9]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        x_fp32 = x.float()
        norm = x_fp32 * torch.rsqrt(x_fp32.pow(2).mean(-1, keepdim=True) + self.eps)
        return (norm * self.weight).type_as(x)

In [10]:
a_rms = RMSNorm(embed_dim)
a_rms.weight

Parameter containing:
tensor([1., 1., 1., 1.], requires_grad=True)

In [11]:
action_emb = a_rms(action_emb)
action_emb.shape, action_emb

(torch.Size([2, 2, 4]),
 tensor([[[0.3651, 0.7303, 1.0954, 1.4606],
          [0.3651, 0.7303, 1.0954, 1.4606]],
 
         [[0.3651, 0.7303, 1.0954, 1.4606],
          [0.3651, 0.7303, 1.0954, 1.4606]]], grad_fn=<MulBackward0>))

#### Adapter - GELU Nonlinearity

Now we're ready for our non-linearity. The [GELU](https://docs.pytorch.org/docs/stable/generated/torch.nn.GELU.html) function is approximately linear above 1 and pulls most values below -2 to 0. Between -2 and 0, most values are pulled closer to 0 and there's a slight non-linearity between 0 and 1. After RMSNorm, our values have an RMS magnitude close to 1, so you'll see GELU introduce a meaningful non-linearity. Small positive values will be slightly reduced while larger values pass through nearly unchanged.

In [12]:
a_gelu = nn.GELU()

action_emb = a_gelu(action_emb)
action_emb.shape, action_emb

(torch.Size([2, 2, 4]),
 tensor([[[0.2346, 0.5604, 0.9457, 1.3553],
          [0.2346, 0.5604, 0.9457, 1.3553]],
 
         [[0.2346, 0.5604, 0.9457, 1.3553],
          [0.2346, 0.5604, 0.9457, 1.3553]]], grad_fn=<GeluBackward0>))

#### Adapter - Linear Layer 2

Now that we've done our nonlinearity, we'll do a final layer to let the model learn how the embedding dimensions relate to one another.

In [13]:
a_l2 = nn.Linear(embed_dim, embed_dim)
vs, d = embed_dim, embed_dim
rows = torch.full((vs,), 1.0).unsqueeze(0)
cols = torch.arange(d).unsqueeze(1)
pattern = 1 * (rows + 1 * cols)
a_l2.weight = nn.Parameter(pattern)
nn.init.zeros_(a_l2.bias)
a_l2.weight, a_l2.bias

(Parameter containing:
 tensor([[1., 1., 1., 1.],
         [2., 2., 2., 2.],
         [3., 3., 3., 3.],
         [4., 4., 4., 4.]], requires_grad=True),
 Parameter containing:
 tensor([0., 0., 0., 0.], requires_grad=True))

In [14]:
action_emb = a_l2(action_emb)
action_emb.shape, action_emb

(torch.Size([2, 2, 4]),
 tensor([[[ 3.0961,  6.1922,  9.2883, 12.3844],
          [ 3.0961,  6.1922,  9.2883, 12.3845]],
 
         [[ 3.0961,  6.1922,  9.2883, 12.3844],
          [ 3.0961,  6.1922,  9.2883, 12.3844]]], grad_fn=<ViewBackward0>))

### Target Gene Embedding Projection

Now we'll create the learnable target embedding tokens for each gene position. This will help ensure that if we only want a subset of genes predicted, we can do it reliably. We sometimes call these embeddings queries since they act like "search requests" telling the model: for each gene, predict what its representation will look like after perturbation.

In [15]:
mask_queries = nn.Embedding(num_genes, embed_dim)
d, vs = num_genes, embed_dim
cols = torch.arange(d).unsqueeze(1)
rows = torch.arange(vs).unsqueeze(0)
pattern = 1.0 * (rows + cols)
mask_queries.weight = nn.Parameter(pattern)
mask_queries.weight

Parameter containing:
tensor([[ 0.,  1.,  2.,  3.],
        [ 1.,  2.,  3.,  4.],
        [ 2.,  3.,  4.,  5.],
        [ 3.,  4.,  5.,  6.],
        [ 4.,  5.,  6.,  7.],
        [ 5.,  6.,  7.,  8.],
        [ 6.,  7.,  8.,  9.],
        [ 7.,  8.,  9., 10.]], requires_grad=True)

In [16]:
queries = mask_queries(target_indices)
queries

tensor([[[ 0.,  1.,  2.,  3.],
         [ 1.,  2.,  3.,  4.],
         [ 2.,  3.,  4.,  5.],
         [ 3.,  4.,  5.,  6.],
         [ 4.,  5.,  6.,  7.],
         [ 5.,  6.,  7.,  8.],
         [ 6.,  7.,  8.,  9.],
         [ 7.,  8.,  9., 10.]],

        [[ 0.,  1.,  2.,  3.],
         [ 1.,  2.,  3.,  4.],
         [ 2.,  3.,  4.,  5.],
         [ 3.,  4.,  5.,  6.],
         [ 4.,  5.,  6.,  7.],
         [ 5.,  6.,  7.,  8.],
         [ 6.,  7.,  8.,  9.],
         [ 7.,  8.,  9., 10.]]], grad_fn=<EmbeddingBackward0>)

#### Project Cell State To Predictor Space

Before we can combine our cell state and target queries, we need to project our control cell state into the same embedding dimensions as our predictor. We do this with a simple linear layer.

In [17]:
in_proj = nn.Linear(context_dim, embed_dim)
context_latents = in_proj(context_latents)
context_latents.shape, context_latents

(torch.Size([2, 8, 4]),
 tensor([[[ 0.9260, -0.5178, -0.6042,  2.1072],
          [-1.0750, -1.0783, -2.5746,  3.9041],
          [-0.2064, -0.3473, -0.0395,  2.1415],
          [-0.5179, -0.7089, -2.8189,  4.2011],
          [ 1.7408, -1.4990, -2.1448,  2.9720],
          [-0.9643,  0.2481, -0.4558,  1.8468],
          [-0.0560, -2.5143, -2.3377,  4.2061],
          [ 0.2395, -1.5844,  1.6070,  2.0200]],
 
         [[-0.0507, -1.5778, -2.3677,  4.9429],
          [ 3.0169, -0.2014,  1.0213,  0.2960],
          [ 0.4950, -0.6619,  0.4367,  1.7346],
          [-0.1149, -1.9085, -0.2686,  4.3146],
          [ 0.5246, -0.7091, -0.1565,  1.7632],
          [ 0.5818, -1.0127, -2.7631,  5.3203],
          [ 1.4450, -1.3709, -2.1276,  3.0808],
          [ 1.6148,  1.2113, -3.4417,  2.5541]]], grad_fn=<ViewBackward0>))

#### Concatenate Cell State and Target

Now that we have our target embeddings identified, we'll concatenate them with the context latent into one representation. This representation will double the token dimension (dim=1). This model will use attention and other layers to update the target based on the context latents. At the end, we'll slice out the target and use that for our model output.

*This is the same pattern used in masked token prediction: visible tokens (context) and mask tokens (queries) are concatenated, attention mixes information, and then the mask positions are read off.*

In [18]:
sequence = torch.cat([context_latents, queries], dim=1)
sequence.shape, sequence

(torch.Size([2, 16, 4]),
 tensor([[[ 0.9260, -0.5178, -0.6042,  2.1072],
          [-1.0750, -1.0783, -2.5746,  3.9041],
          [-0.2064, -0.3473, -0.0395,  2.1415],
          [-0.5179, -0.7089, -2.8189,  4.2011],
          [ 1.7408, -1.4990, -2.1448,  2.9720],
          [-0.9643,  0.2481, -0.4558,  1.8468],
          [-0.0560, -2.5143, -2.3377,  4.2061],
          [ 0.2395, -1.5844,  1.6070,  2.0200],
          [ 0.0000,  1.0000,  2.0000,  3.0000],
          [ 1.0000,  2.0000,  3.0000,  4.0000],
          [ 2.0000,  3.0000,  4.0000,  5.0000],
          [ 3.0000,  4.0000,  5.0000,  6.0000],
          [ 4.0000,  5.0000,  6.0000,  7.0000],
          [ 5.0000,  6.0000,  7.0000,  8.0000],
          [ 6.0000,  7.0000,  8.0000,  9.0000],
          [ 7.0000,  8.0000,  9.0000, 10.0000]],
 
         [[-0.0507, -1.5778, -2.3677,  4.9429],
          [ 3.0169, -0.2014,  1.0213,  0.2960],
          [ 0.4950, -0.6619,  0.4367,  1.7346],
          [-0.1149, -1.9085, -0.2686,  4.3146],
          [ 

### Multilayer Transformer Block

The transformer block for our predictor still outputs a latent representation but has a different structure from the cell state encoder transformer block. Since we're combining the perturbation and the context latents, this transformer will first use cross-attention and then use self-attention on the output. The cross-attention allows the predictor to inject information from the perturbation action embedding (via cross-attention) into the cell state and then let that information propagate across all gene positions (via self-attention).

The predictor block consists of 6 layers:
1. RMSNorm 1
2. Multi-headed linear cross-attention (mechanism injection)
3. RMSNorm 2
4. Multi-headed gated linear self-attention (dynamics propagation)
5. RMSNorm 3
6. SwiGLU MLP

Residual connections provide gradient bypassing around each of the three sub-layers. This set of layers is repeated based on how many layers are configured.

#### RMSNorm 1

We'll start with our first normalization before the cross-attention layer. We'll only normalize the sequence data, not the perturbation, as that is the content that is updated. You'll see this pattern show up in the residual connection also later.

In [19]:
x = sequence
x.shape, x

(torch.Size([2, 16, 4]),
 tensor([[[ 0.9260, -0.5178, -0.6042,  2.1072],
          [-1.0750, -1.0783, -2.5746,  3.9041],
          [-0.2064, -0.3473, -0.0395,  2.1415],
          [-0.5179, -0.7089, -2.8189,  4.2011],
          [ 1.7408, -1.4990, -2.1448,  2.9720],
          [-0.9643,  0.2481, -0.4558,  1.8468],
          [-0.0560, -2.5143, -2.3377,  4.2061],
          [ 0.2395, -1.5844,  1.6070,  2.0200],
          [ 0.0000,  1.0000,  2.0000,  3.0000],
          [ 1.0000,  2.0000,  3.0000,  4.0000],
          [ 2.0000,  3.0000,  4.0000,  5.0000],
          [ 3.0000,  4.0000,  5.0000,  6.0000],
          [ 4.0000,  5.0000,  6.0000,  7.0000],
          [ 5.0000,  6.0000,  7.0000,  8.0000],
          [ 6.0000,  7.0000,  8.0000,  9.0000],
          [ 7.0000,  8.0000,  9.0000, 10.0000]],
 
         [[-0.0507, -1.5778, -2.3677,  4.9429],
          [ 3.0169, -0.2014,  1.0213,  0.2960],
          [ 0.4950, -0.6619,  0.4367,  1.7346],
          [-0.1149, -1.9085, -0.2686,  4.3146],
          [ 

In [20]:
rms1 = RMSNorm(embed_dim)
rms1.weight

Parameter containing:
tensor([1., 1., 1., 1.], requires_grad=True)

In [21]:
x_norm1 = rms1(x)
x_norm1.shape, x_norm1

(torch.Size([2, 16, 4]),
 tensor([[[ 0.7605, -0.4252, -0.4962,  1.7305],
          [-0.4371, -0.4385, -1.0470,  1.5876],
          [-0.1894, -0.3187, -0.0363,  1.9650],
          [-0.2017, -0.2761, -1.0980,  1.6363],
          [ 0.8049, -0.6931, -0.9917,  1.3742],
          [-0.8982,  0.2311, -0.4246,  1.7203],
          [-0.0206, -0.9261, -0.8611,  1.5493],
          [ 0.1577, -1.0430,  1.0578,  1.3298],
          [ 0.0000,  0.5345,  1.0690,  1.6036],
          [ 0.3651,  0.7303,  1.0954,  1.4606],
          [ 0.5443,  0.8165,  1.0887,  1.3608],
          [ 0.6470,  0.8627,  1.0783,  1.2940],
          [ 0.7127,  0.8909,  1.0690,  1.2472],
          [ 0.7581,  0.9097,  1.0613,  1.2130],
          [ 0.7913,  0.9231,  1.0550,  1.1869],
          [ 0.8165,  0.9331,  1.0498,  1.1664]],
 
         [[-0.0178, -0.5533, -0.8302,  1.7333],
          [ 1.8825, -0.1257,  0.6373,  0.1847],
          [ 0.5024, -0.6718,  0.4433,  1.7606],
          [-0.0486, -0.8075, -0.1137,  1.8256],
          [ 

#### Multi-Headed Linear Cross-Attention

Next we'll run our cross-attention to allow updates from the perturbation. In the cross-attention layer, the query comes from the gene sequence (context + target) and the key/value come from the MLP-processed action embedding. With this attention, each gene position can look at the perturbation tokens and pull in relevant information, injecting the perturbation information into the gene representations. We apply the following calculation.

$$\begin{aligned}
Q &= \text{elu}(x_\text{norm}\, W_q^\top + b_q) + 1.0 \\
K &= \text{elu}(\text{action\_emb}\, W_k^\top + b_k) + 1.0 \\
V &= \text{action\_emb}\, W_v^\top + b_v \\
\\
\text{Attn} &= \frac{Q (K^\top V)}{Q (\sum_{j} K_j)^\top + \epsilon} \\
\\
y &= \text{Attn}\, W_c^\top + b_c
\end{aligned}$$


*Since the key and value are based on the perturbation, we do not add in gating like we do with self-attention. The model relies on the learned projections and the subsequent self-attention layer to regulate information flow.*

In [22]:
B, T_q, C = x_norm1.size()
head_dim = embed_dim // heads
B, T_q, C, head_dim

(2, 16, 4, 2)

**Cross-attention**

For cross-attention, the key/value input comes from the action embedding rather than the sequence itself. This builds a relationship between each gene position and the perturbation tokens. You'll notice that `kv_input` has a shape based on the perturbation batch size and n_perts, and that the cross-attention allows the cell state to pull from all of the cell's perturbations.

In [23]:
kv_input = action_emb
kv_input.shape, kv_input

(torch.Size([2, 2, 4]),
 tensor([[[ 3.0961,  6.1922,  9.2883, 12.3844],
          [ 3.0961,  6.1922,  9.2883, 12.3845]],
 
         [[ 3.0961,  6.1922,  9.2883, 12.3844],
          [ 3.0961,  6.1922,  9.2883, 12.3844]]], grad_fn=<ViewBackward0>))

In [24]:
T_kv = kv_input.size(1)
T_kv

2

**Query**

Now let's calculate our query. The query is the current position's "search request." Every layer and head issues queries that look for relationships with the key/value source. Our query first calculates a linear weight $Q=x_{norm}W^\top+b$, resulting in a vector of $[B,T_q,C]$ where $T_q$ is our gene sequence length. We then split $C$, the embedding dimension, across our $H$ heads and transpose the middle dimensions, resulting in a tensor of $[B,Heads,T_q,C_{Heads}]$.

Finally, we'll apply the ELU+1 function. In linear attention, since we're skipping softmax, we need to ensure that the attention weights stay non-negative like they would with softmax. ELU+1 approximates softmax attention's behavior while keeping the linear complexity benefit.

In [25]:
q_proj = nn.Linear(embed_dim, embed_dim)
nn.init.constant_(q_proj.weight, -0.1)
nn.init.constant_(q_proj.bias, 0)
q_proj.weight

Parameter containing:
tensor([[-0.1000, -0.1000, -0.1000, -0.1000],
        [-0.1000, -0.1000, -0.1000, -0.1000],
        [-0.1000, -0.1000, -0.1000, -0.1000],
        [-0.1000, -0.1000, -0.1000, -0.1000]], requires_grad=True)

In [26]:
q = q_proj(x_norm1).view(B, T_q, heads, head_dim).transpose(1, 2)
q.shape, q

(torch.Size([2, 2, 16, 2]),
 tensor([[[[-0.1570, -0.1570],
           [ 0.0335,  0.0335],
           [-0.1421, -0.1421],
           [-0.0061, -0.0061],
           [-0.0494, -0.0494],
           [-0.0629, -0.0629],
           [ 0.0259,  0.0259],
           [-0.1502, -0.1502],
           [-0.3207, -0.3207],
           [-0.3651, -0.3651],
           [-0.3810, -0.3810],
           [-0.3882, -0.3882],
           [-0.3920, -0.3920],
           [-0.3942, -0.3942],
           [-0.3956, -0.3956],
           [-0.3966, -0.3966]],
 
          [[-0.1570, -0.1570],
           [ 0.0335,  0.0335],
           [-0.1421, -0.1421],
           [-0.0061, -0.0061],
           [-0.0494, -0.0494],
           [-0.0629, -0.0629],
           [ 0.0259,  0.0259],
           [-0.1502, -0.1502],
           [-0.3207, -0.3207],
           [-0.3651, -0.3651],
           [-0.3810, -0.3810],
           [-0.3882, -0.3882],
           [-0.3920, -0.3920],
           [-0.3942, -0.3942],
           [-0.3956, -0.3956],
        

In [27]:
q = F.elu(q) + 1.0
q.shape, q

(torch.Size([2, 2, 16, 2]),
 tensor([[[[0.8547, 0.8547],
           [1.0335, 1.0335],
           [0.8676, 0.8676],
           [0.9940, 0.9940],
           [0.9518, 0.9518],
           [0.9391, 0.9391],
           [1.0259, 1.0259],
           [0.8605, 0.8605],
           [0.7256, 0.7256],
           [0.6941, 0.6941],
           [0.6832, 0.6832],
           [0.6783, 0.6783],
           [0.6757, 0.6757],
           [0.6742, 0.6742],
           [0.6733, 0.6733],
           [0.6726, 0.6726]],
 
          [[0.8547, 0.8547],
           [1.0335, 1.0335],
           [0.8676, 0.8676],
           [0.9940, 0.9940],
           [0.9518, 0.9518],
           [0.9391, 0.9391],
           [1.0259, 1.0259],
           [0.8605, 0.8605],
           [0.7256, 0.7256],
           [0.6941, 0.6941],
           [0.6832, 0.6832],
           [0.6783, 0.6783],
           [0.6757, 0.6757],
           [0.6742, 0.6742],
           [0.6733, 0.6733],
           [0.6726, 0.6726]]],
 
 
         [[[0.9673, 0.9673],
      

**Key**

Next, we calculate the key. The key acts as a matching tag/address for each token in the key/value source. It is compared with the query to produce relevance scores. Since this is cross-attention, K is projected from the action embedding, allowing the model to build relationships between genes and perturbations.

We calculate a linear weight $K=\text{action\_emb}\cdot W^\top+b$, resulting in a vector of $[B,T_{kv},C]$ where $T_{kv}$ is the number of perturbation tokens. We then split $C$, the embedding dimension, across our $H$ heads and transpose the middle dimensions, resulting in a tensor of $[B,Heads,T_{kv},C_{Heads}]$.

Finally, we'll apply the ELU+1 function. In linear attention, since we're skipping softmax, we need to ensure that the attention weights stay non-negative like they would with softmax. ELU+1 approximates softmax attention's behavior while keeping the linear complexity benefit.

In [28]:
k_proj = nn.Linear(embed_dim, embed_dim)
nn.init.constant_(k_proj.weight, 0.2)
nn.init.constant_(k_proj.bias, 0)
k_proj.weight

Parameter containing:
tensor([[0.2000, 0.2000, 0.2000, 0.2000],
        [0.2000, 0.2000, 0.2000, 0.2000],
        [0.2000, 0.2000, 0.2000, 0.2000],
        [0.2000, 0.2000, 0.2000, 0.2000]], requires_grad=True)

In [29]:
k = k_proj(kv_input).view(B, T_kv, heads, head_dim).transpose(1, 2)
k.shape, k

(torch.Size([2, 2, 2, 2]),
 tensor([[[[6.1922, 6.1922],
           [6.1922, 6.1922]],
 
          [[6.1922, 6.1922],
           [6.1922, 6.1922]]],
 
 
         [[[6.1922, 6.1922],
           [6.1922, 6.1922]],
 
          [[6.1922, 6.1922],
           [6.1922, 6.1922]]]], grad_fn=<TransposeBackward0>))

In [30]:
k = F.elu(k) + 1.0
k

tensor([[[[7.1922, 7.1922],
          [7.1922, 7.1922]],

         [[7.1922, 7.1922],
          [7.1922, 7.1922]]],


        [[[7.1922, 7.1922],
          [7.1922, 7.1922]],

         [[7.1922, 7.1922],
          [7.1922, 7.1922]]]], grad_fn=<AddBackward0>)

**Value**

Next, we calculate the value. The value is the payload you actually mix in once something matches. For cross-attention, it's a learned projection of the action embedding so the model can copy the right kind of perturbation information into the gene representations.

We calculate a linear weight $V=\text{action\_emb}\cdot W^\top+b$, resulting in a vector of $[B,T_{kv},C]$. We then split $C$, the embedding dimension, across our $H$ heads and transpose the middle dimensions, resulting in a tensor of $[B,Heads,T_{kv},C_{Heads}]$.

For V, we do not use ELU+1 since the QK product will act on V.

In [31]:
v_proj = nn.Linear(embed_dim, embed_dim)
nn.init.constant_(v_proj.weight, 1.0)
nn.init.constant_(v_proj.bias, 0)
v_proj.weight

Parameter containing:
tensor([[1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.]], requires_grad=True)

In [32]:
v = v_proj(kv_input).view(B, T_kv, heads, head_dim).transpose(1, 2)
v.shape, v

(torch.Size([2, 2, 2, 2]),
 tensor([[[[30.9611, 30.9611],
           [30.9611, 30.9611]],
 
          [[30.9611, 30.9611],
           [30.9611, 30.9611]]],
 
 
         [[[30.9611, 30.9611],
           [30.9611, 30.9611]],
 
          [[30.9611, 30.9611],
           [30.9611, 30.9611]]]], grad_fn=<TransposeBackward0>))

**Normalization denominator**

In attention, when using softmax, attention values become probabilities and sum to 1. With linear attention, we need to manually do this normalization.

In softmax attention, the softmax inherently normalizes so weights sum to 1. Linear attention doesn't have that, so we need to create the denominator $z$. We'll do this by summing the tokens per embedding dimension, resulting in a $[B,Heads,C_{Heads},1]$ dimension that we can then use in the denominator of our attention calculation.

In [33]:
k_sum = k.sum(dim=-2).unsqueeze(-1)
k_sum.shape, k_sum

(torch.Size([2, 2, 2, 1]),
 tensor([[[[14.3845],
           [14.3845]],
 
          [[14.3845],
           [14.3845]]],
 
 
         [[[14.3844],
           [14.3844]],
 
          [[14.3844],
           [14.3844]]]], grad_fn=<UnsqueezeBackward0>))

**Denominator**

We're now ready to calculate the rest of the denominator. The denominator is the sum of attention weights for each query across all keys. Each query gets its own normalizing constant, so queries attending to high-magnitude keys don't get inflated outputs. We add a final epsilon to ensure the denominator is not zero. The result is a $[B, Heads, T, 1]$ tensor.

In [34]:
z = 1.0 / (q @ k_sum + 1e-6)
z.shape, z

(torch.Size([2, 2, 16, 1]),
 tensor([[[[0.0407],
           [0.0336],
           [0.0401],
           [0.0350],
           [0.0365],
           [0.0370],
           [0.0339],
           [0.0404],
           [0.0479],
           [0.0501],
           [0.0509],
           [0.0512],
           [0.0514],
           [0.0516],
           [0.0516],
           [0.0517]],
 
          [[0.0407],
           [0.0336],
           [0.0401],
           [0.0350],
           [0.0365],
           [0.0370],
           [0.0339],
           [0.0404],
           [0.0479],
           [0.0501],
           [0.0509],
           [0.0512],
           [0.0514],
           [0.0516],
           [0.0516],
           [0.0517]]],
 
 
         [[[0.0359],
           [0.0450],
           [0.0426],
           [0.0379],
           [0.0401],
           [0.0373],
           [0.0365],
           [0.0377],
           [0.0479],
           [0.0501],
           [0.0509],
           [0.0512],
           [0.0514],
           [0.0516

**Numerator**

Now we're ready to complete our numerator. This is just a matter of multiplying Q, K, and V. We'll need to transpose K to get the interaction between the query and key to then multiply against the value.

Since we're looking to save memory, we actually first multiply the key and value to create head-dimension matrices, and then multiply by the query. This order of operations is part of what saves memory.

In [35]:
kv = k.transpose(-2, -1) @ v
kv.shape, kv

(torch.Size([2, 2, 2, 2]),
 tensor([[[[445.3588, 445.3588],
           [445.3588, 445.3588]],
 
          [[445.3588, 445.3588],
           [445.3588, 445.3588]]],
 
 
         [[[445.3587, 445.3587],
           [445.3587, 445.3587]],
 
          [[445.3587, 445.3587],
           [445.3587, 445.3587]]]], grad_fn=<UnsafeViewBackward0>))

In [36]:
qkv = q @ kv
qkv.shape, qkv

(torch.Size([2, 2, 16, 2]),
 tensor([[[[761.3309, 761.3309],
           [920.5566, 920.5566],
           [772.7534, 772.7534],
           [885.3430, 885.3430],
           [847.7593, 847.7593],
           [836.4470, 836.4470],
           [913.7479, 913.7479],
           [766.4698, 766.4698],
           [646.3323, 646.3323],
           [618.2415, 618.2415],
           [608.4992, 608.4992],
           [604.1541, 604.1541],
           [601.8716, 601.8716],
           [600.5323, 600.5323],
           [599.6817, 599.6817],
           [599.1088, 599.1088]],
 
          [[761.3309, 761.3309],
           [920.5566, 920.5566],
           [772.7534, 772.7534],
           [885.3430, 885.3430],
           [847.7593, 847.7593],
           [836.4470, 836.4470],
           [913.7479, 913.7479],
           [766.4698, 766.4698],
           [646.3323, 646.3323],
           [618.2415, 618.2415],
           [608.4992, 608.4992],
           [604.1541, 604.1541],
           [601.8716, 601.8716],
           [

**Linear Attention**

Now we're ready to normalize our numerator by the denominator. You'll see that because of our extremely consistent values and initialization, we're ending up with very consistent values by example even across heads.

In [37]:
y = qkv * z
y.shape, y

(torch.Size([2, 2, 16, 2]),
 tensor([[[[30.9611, 30.9611],
           [30.9611, 30.9611],
           [30.9611, 30.9611],
           [30.9611, 30.9611],
           [30.9611, 30.9611],
           [30.9611, 30.9611],
           [30.9611, 30.9611],
           [30.9611, 30.9611],
           [30.9611, 30.9611],
           [30.9611, 30.9611],
           [30.9611, 30.9611],
           [30.9611, 30.9611],
           [30.9611, 30.9611],
           [30.9611, 30.9611],
           [30.9611, 30.9611],
           [30.9611, 30.9611]],
 
          [[30.9611, 30.9611],
           [30.9611, 30.9611],
           [30.9611, 30.9611],
           [30.9611, 30.9611],
           [30.9611, 30.9611],
           [30.9611, 30.9611],
           [30.9611, 30.9611],
           [30.9611, 30.9611],
           [30.9611, 30.9611],
           [30.9611, 30.9611],
           [30.9611, 30.9611],
           [30.9611, 30.9611],
           [30.9611, 30.9611],
           [30.9611, 30.9611],
           [30.9611, 30.9611],
        

**Collapse heads**

We can now bring our heads back together. We have to undo our head splitting. First we flip our heads and tokens (genes) so that we have a $[B,T,Heads,C_{Heads}]$, and then we collapse the head and head embeddings to end up with $[B,T,C]$.

In [38]:
y = y.transpose(1, 2).contiguous().view(B, T_q, C)
y.shape, y

(torch.Size([2, 16, 4]),
 tensor([[[30.9611, 30.9611, 30.9611, 30.9611],
          [30.9611, 30.9611, 30.9611, 30.9611],
          [30.9611, 30.9611, 30.9611, 30.9611],
          [30.9611, 30.9611, 30.9611, 30.9611],
          [30.9611, 30.9611, 30.9611, 30.9611],
          [30.9611, 30.9611, 30.9611, 30.9611],
          [30.9611, 30.9611, 30.9611, 30.9611],
          [30.9611, 30.9611, 30.9611, 30.9611],
          [30.9611, 30.9611, 30.9611, 30.9611],
          [30.9611, 30.9611, 30.9611, 30.9611],
          [30.9611, 30.9611, 30.9611, 30.9611],
          [30.9611, 30.9611, 30.9611, 30.9611],
          [30.9611, 30.9611, 30.9611, 30.9611],
          [30.9611, 30.9611, 30.9611, 30.9611],
          [30.9611, 30.9611, 30.9611, 30.9611],
          [30.9611, 30.9611, 30.9611, 30.9611]],
 
         [[30.9611, 30.9611, 30.9611, 30.9611],
          [30.9611, 30.9611, 30.9611, 30.9611],
          [30.9611, 30.9611, 30.9611, 30.9611],
          [30.9611, 30.9611, 30.9611, 30.9611],
          [3

**Cross-head final projection**

Finally, we project the attention output through a final linear layer. This allows the model to learn how to combine information across the different heads.

*Notice that unlike self-attention, we skip the sigmoid gating step here. Cross-attention doesn't apply gating because the model regulates information flow through the learned Q/K/V projections and the downstream self-attention layer's gate.*

In [39]:
c_proj = nn.Linear(embed_dim, embed_dim)
vs, d = embed_dim, embed_dim
rows = torch.full((vs,), 0.1).unsqueeze(0)
cols = torch.arange(d).unsqueeze(1)
pattern = 1 * (rows + 0.01 * cols)

c_proj.weight = nn.Parameter(pattern)
c_proj.weight

Parameter containing:
tensor([[0.1000, 0.1000, 0.1000, 0.1000],
        [0.1100, 0.1100, 0.1100, 0.1100],
        [0.1200, 0.1200, 0.1200, 0.1200],
        [0.1300, 0.1300, 0.1300, 0.1300]], requires_grad=True)

In [40]:
x_attn_cross = c_proj(y)
x_attn_cross.shape, x_attn_cross

(torch.Size([2, 16, 4]),
 tensor([[[12.2888, 14.0559, 15.1641, 16.3969],
          [12.2888, 14.0559, 15.1641, 16.3969],
          [12.2888, 14.0559, 15.1641, 16.3969],
          [12.2888, 14.0559, 15.1641, 16.3969],
          [12.2888, 14.0559, 15.1641, 16.3969],
          [12.2888, 14.0559, 15.1641, 16.3969],
          [12.2888, 14.0559, 15.1641, 16.3969],
          [12.2888, 14.0559, 15.1641, 16.3969],
          [12.2888, 14.0559, 15.1641, 16.3969],
          [12.2888, 14.0559, 15.1641, 16.3969],
          [12.2888, 14.0559, 15.1641, 16.3969],
          [12.2888, 14.0559, 15.1641, 16.3969],
          [12.2888, 14.0559, 15.1641, 16.3969],
          [12.2888, 14.0559, 15.1641, 16.3969],
          [12.2888, 14.0559, 15.1641, 16.3969],
          [12.2888, 14.0559, 15.1641, 16.3969]],
 
         [[12.2888, 14.0559, 15.1641, 16.3969],
          [12.2888, 14.0559, 15.1641, 16.3969],
          [12.2888, 14.0559, 15.1641, 16.3969],
          [12.2888, 14.0559, 15.1641, 16.3969],
          [1

#### Residual Connection

We now have our linear cross-attention calculated and will use a residual connection to allow gradients to bypass the cross-attention layer. You'll see how much larger the residual connection impact is on our output compared to the cross-attention contribution.

For this residual connection, we add the cross-attention update back to the cell-state sequence `x`. The `action_latents` supply the keys and values used to calculate the update, but they are not part of the sequence being updated. Gradients still flow through the action path during training, allowing the adapter and Action Composer to learn how the perturbation should affect the prediction.

In [41]:
x = x + x_attn_cross
x.shape, x

(torch.Size([2, 16, 4]),
 tensor([[[13.2148, 13.5381, 14.5599, 18.5042],
          [11.2138, 12.9776, 12.5895, 20.3010],
          [12.0824, 13.7086, 15.1246, 18.5385],
          [11.7709, 13.3470, 12.3452, 20.5980],
          [14.0296, 12.5569, 13.0193, 19.3689],
          [11.3245, 14.3040, 14.7083, 18.2438],
          [12.2327, 11.5416, 12.8264, 20.6030],
          [12.5283, 12.4715, 16.7711, 18.4170],
          [12.2888, 15.0559, 17.1641, 19.3969],
          [13.2888, 16.0559, 18.1641, 20.3969],
          [14.2888, 17.0559, 19.1641, 21.3969],
          [15.2888, 18.0559, 20.1641, 22.3969],
          [16.2888, 19.0559, 21.1641, 23.3969],
          [17.2888, 20.0559, 22.1641, 24.3969],
          [18.2888, 21.0559, 23.1641, 25.3969],
          [19.2888, 22.0559, 24.1641, 26.3969]],
 
         [[12.2381, 12.4781, 12.7964, 21.3399],
          [15.3057, 13.8545, 16.1854, 16.6929],
          [12.7838, 13.3940, 15.6008, 18.1316],
          [12.1739, 12.1474, 14.8955, 20.7115],
          [1

#### RMSNorm 2

Now that we've calculated the cross-attention, we'll do another round of normalization before the self-attention layer. We'll use RMSNorm again.

In [42]:
rms2 = RMSNorm(embed_dim)
rms2.weight

Parameter containing:
tensor([1., 1., 1., 1.], requires_grad=True)

In [43]:
x_norm2 = rms2(x)
x_norm2.shape, x_norm2

(torch.Size([2, 16, 4]),
 tensor([[[0.8750, 0.8964, 0.9641, 1.2253],
          [0.7627, 0.8826, 0.8562, 1.3807],
          [0.8027, 0.9107, 1.0048, 1.2316],
          [0.7876, 0.8931, 0.8261, 1.3783],
          [0.9357, 0.8375, 0.8684, 1.2919],
          [0.7626, 0.9633, 0.9905, 1.2286],
          [0.8286, 0.7818, 0.8688, 1.3955],
          [0.8203, 0.8166, 1.0981, 1.2059],
          [0.7590, 0.9299, 1.0601, 1.1980],
          [0.7736, 0.9347, 1.0574, 1.1874],
          [0.7865, 0.9388, 1.0549, 1.1778],
          [0.7981, 0.9425, 1.0526, 1.1691],
          [0.8085, 0.9458, 1.0504, 1.1612],
          [0.8178, 0.9487, 1.0484, 1.1541],
          [0.8263, 0.9514, 1.0466, 1.1475],
          [0.8341, 0.9537, 1.0449, 1.1414]],
 
         [[0.8049, 0.8207, 0.8417, 1.4036],
          [0.9845, 0.8911, 1.0411, 1.0737],
          [0.8453, 0.8856, 1.0315, 1.1988],
          [0.7914, 0.7896, 0.9683, 1.3463],
          [0.8555, 0.8911, 1.0020, 1.2125],
          [0.8303, 0.8414, 0.8000, 1.4010],
    

#### Multi-Headed Gated Linear Self-Attention

Now that the perturbation information has been injected via cross-attention, we use self-attention to let that information propagate across all gene positions. The difference with this layer is that each position in the sequence (both context genes and target genes) can attend to every other position, allowing the model to learn how the perturbation effect spreads across the gene network.

The main differences here are that the kv_input is the sequence and we add in gating. The gating at the end allows the model to determine how much the attention should impact each specific position.

Our gated linear attention has the following update to the calculation:

$$\begin{aligned}
\text{elu}(x) &= \begin{cases} x & \text{if } x > 0 \\ \alpha (e^x - 1) & \text{if } x \le 0 \end{cases} \\
\\
Q &= \text{elu}(x W_q^\top + b_q) + 1.0 \\
K &= \text{elu}(x W_k^\top + b_k) + 1.0 \\
V &= x W_v^\top + b_v \\
\\
\text{Attn} &= \frac{Q (K^\top V)}{Q (\sum_{j} K_j)^\top + \epsilon} \\
\text{Gate} &= \sigma(x W_{gate}^\top + b_{gate}) \\
\\
y &= \left( \text{Gate} \odot \text{Attn} \right) W_c^\top + b_c
\end{aligned}$$

*Note that I'll mainly call out unique calculations for self-attention and won't repeat what we've explained in cross-attention*

In [44]:
B, T_q, C = x_norm2.size()
head_dim = embed_dim // heads
B, T_q, C, head_dim

(2, 16, 4, 2)

**Self-attention**

For self-attention, the key/value source is the same normalized output from the cross-attention as the query source. Using the same key/value and query source allows each position (both context genes and query genes) to build relationships with every other position.

*You'll notice most of the code looks the same since we built 1 attention class to handle self- and cross-attention.*

In [45]:
kv_input = x_norm2
kv_input

tensor([[[0.8750, 0.8964, 0.9641, 1.2253],
         [0.7627, 0.8826, 0.8562, 1.3807],
         [0.8027, 0.9107, 1.0048, 1.2316],
         [0.7876, 0.8931, 0.8261, 1.3783],
         [0.9357, 0.8375, 0.8684, 1.2919],
         [0.7626, 0.9633, 0.9905, 1.2286],
         [0.8286, 0.7818, 0.8688, 1.3955],
         [0.8203, 0.8166, 1.0981, 1.2059],
         [0.7590, 0.9299, 1.0601, 1.1980],
         [0.7736, 0.9347, 1.0574, 1.1874],
         [0.7865, 0.9388, 1.0549, 1.1778],
         [0.7981, 0.9425, 1.0526, 1.1691],
         [0.8085, 0.9458, 1.0504, 1.1612],
         [0.8178, 0.9487, 1.0484, 1.1541],
         [0.8263, 0.9514, 1.0466, 1.1475],
         [0.8341, 0.9537, 1.0449, 1.1414]],

        [[0.8049, 0.8207, 0.8417, 1.4036],
         [0.9845, 0.8911, 1.0411, 1.0737],
         [0.8453, 0.8856, 1.0315, 1.1988],
         [0.7914, 0.7896, 0.9683, 1.3463],
         [0.8555, 0.8911, 1.0020, 1.2125],
         [0.8303, 0.8414, 0.8000, 1.4010],
         [0.9162, 0.8462, 0.8697, 1.2993],
         

In [46]:
T_kv = kv_input.size(1)
T_kv

16

**Query**

In [47]:
q_proj2 = nn.Linear(embed_dim, embed_dim)
nn.init.constant_(q_proj2.weight, -0.2)
nn.init.constant_(q_proj2.bias, 0)
q_proj2.weight

Parameter containing:
tensor([[-0.2000, -0.2000, -0.2000, -0.2000],
        [-0.2000, -0.2000, -0.2000, -0.2000],
        [-0.2000, -0.2000, -0.2000, -0.2000],
        [-0.2000, -0.2000, -0.2000, -0.2000]], requires_grad=True)

In [48]:
q = q_proj2(x_norm2).view(B, T_q, heads, head_dim).transpose(1, 2)
q.shape, q

(torch.Size([2, 2, 16, 2]),
 tensor([[[[-0.7922, -0.7922],
           [-0.7764, -0.7764],
           [-0.7899, -0.7899],
           [-0.7770, -0.7770],
           [-0.7867, -0.7867],
           [-0.7890, -0.7890],
           [-0.7749, -0.7749],
           [-0.7882, -0.7882],
           [-0.7894, -0.7894],
           [-0.7906, -0.7906],
           [-0.7916, -0.7916],
           [-0.7925, -0.7925],
           [-0.7932, -0.7932],
           [-0.7938, -0.7938],
           [-0.7944, -0.7944],
           [-0.7948, -0.7948]],
 
          [[-0.7922, -0.7922],
           [-0.7764, -0.7764],
           [-0.7899, -0.7899],
           [-0.7770, -0.7770],
           [-0.7867, -0.7867],
           [-0.7890, -0.7890],
           [-0.7749, -0.7749],
           [-0.7882, -0.7882],
           [-0.7894, -0.7894],
           [-0.7906, -0.7906],
           [-0.7916, -0.7916],
           [-0.7925, -0.7925],
           [-0.7932, -0.7932],
           [-0.7938, -0.7938],
           [-0.7944, -0.7944],
        

In [49]:
q = F.elu(q) + 1.0
q.shape, q

(torch.Size([2, 2, 16, 2]),
 tensor([[[[0.4529, 0.4529],
           [0.4600, 0.4600],
           [0.4539, 0.4539],
           [0.4598, 0.4598],
           [0.4553, 0.4553],
           [0.4543, 0.4543],
           [0.4607, 0.4607],
           [0.4547, 0.4547],
           [0.4541, 0.4541],
           [0.4536, 0.4536],
           [0.4531, 0.4531],
           [0.4527, 0.4527],
           [0.4524, 0.4524],
           [0.4521, 0.4521],
           [0.4519, 0.4519],
           [0.4517, 0.4517]],
 
          [[0.4529, 0.4529],
           [0.4600, 0.4600],
           [0.4539, 0.4539],
           [0.4598, 0.4598],
           [0.4553, 0.4553],
           [0.4543, 0.4543],
           [0.4607, 0.4607],
           [0.4547, 0.4547],
           [0.4541, 0.4541],
           [0.4536, 0.4536],
           [0.4531, 0.4531],
           [0.4527, 0.4527],
           [0.4524, 0.4524],
           [0.4521, 0.4521],
           [0.4519, 0.4519],
           [0.4517, 0.4517]]],
 
 
         [[[0.4611, 0.4611],
      

**Key**

The difference here is that the key works on the same data as the query. This is the self-attention component.

In [50]:
k_proj2 = nn.Linear(embed_dim, embed_dim)
nn.init.constant_(k_proj2.weight, -0.05)
nn.init.constant_(k_proj2.bias, 0)
k_proj2.weight

Parameter containing:
tensor([[-0.0500, -0.0500, -0.0500, -0.0500],
        [-0.0500, -0.0500, -0.0500, -0.0500],
        [-0.0500, -0.0500, -0.0500, -0.0500],
        [-0.0500, -0.0500, -0.0500, -0.0500]], requires_grad=True)

In [51]:
k = k_proj2(kv_input).view(B, T_kv, heads, head_dim).transpose(1, 2)
k.shape, k

(torch.Size([2, 2, 16, 2]),
 tensor([[[[-0.1980, -0.1980],
           [-0.1941, -0.1941],
           [-0.1975, -0.1975],
           [-0.1943, -0.1943],
           [-0.1967, -0.1967],
           [-0.1972, -0.1972],
           [-0.1937, -0.1937],
           [-0.1971, -0.1971],
           [-0.1974, -0.1974],
           [-0.1977, -0.1977],
           [-0.1979, -0.1979],
           [-0.1981, -0.1981],
           [-0.1983, -0.1983],
           [-0.1985, -0.1985],
           [-0.1986, -0.1986],
           [-0.1987, -0.1987]],
 
          [[-0.1980, -0.1980],
           [-0.1941, -0.1941],
           [-0.1975, -0.1975],
           [-0.1943, -0.1943],
           [-0.1967, -0.1967],
           [-0.1972, -0.1972],
           [-0.1937, -0.1937],
           [-0.1971, -0.1971],
           [-0.1974, -0.1974],
           [-0.1977, -0.1977],
           [-0.1979, -0.1979],
           [-0.1981, -0.1981],
           [-0.1983, -0.1983],
           [-0.1985, -0.1985],
           [-0.1986, -0.1986],
        

In [52]:
k = F.elu(k) + 1.0
k

tensor([[[[0.8203, 0.8203],
          [0.8236, 0.8236],
          [0.8208, 0.8208],
          [0.8234, 0.8234],
          [0.8215, 0.8215],
          [0.8210, 0.8210],
          [0.8239, 0.8239],
          [0.8211, 0.8211],
          [0.8209, 0.8209],
          [0.8207, 0.8207],
          [0.8205, 0.8205],
          [0.8203, 0.8203],
          [0.8201, 0.8201],
          [0.8200, 0.8200],
          [0.8199, 0.8199],
          [0.8198, 0.8198]],

         [[0.8203, 0.8203],
          [0.8236, 0.8236],
          [0.8208, 0.8208],
          [0.8234, 0.8234],
          [0.8215, 0.8215],
          [0.8210, 0.8210],
          [0.8239, 0.8239],
          [0.8211, 0.8211],
          [0.8209, 0.8209],
          [0.8207, 0.8207],
          [0.8205, 0.8205],
          [0.8203, 0.8203],
          [0.8201, 0.8201],
          [0.8200, 0.8200],
          [0.8199, 0.8199],
          [0.8198, 0.8198]]],


        [[[0.8240, 0.8240],
          [0.8191, 0.8191],
          [0.8203, 0.8203],
          [0.8

**Value**

Like the key, the value also works on the same data as the query. This is also part of the self-attention component.

In [53]:
v_proj2 = nn.Linear(embed_dim, embed_dim)
nn.init.constant_(v_proj2.weight, 0.9)
nn.init.constant_(v_proj2.bias, 0)
v_proj2.weight

Parameter containing:
tensor([[0.9000, 0.9000, 0.9000, 0.9000],
        [0.9000, 0.9000, 0.9000, 0.9000],
        [0.9000, 0.9000, 0.9000, 0.9000],
        [0.9000, 0.9000, 0.9000, 0.9000]], requires_grad=True)

In [54]:
v = v_proj2(kv_input).view(B, T_kv, heads, head_dim).transpose(1, 2)
v.shape, v

(torch.Size([2, 2, 16, 2]),
 tensor([[[[3.5647, 3.5647],
           [3.4939, 3.4939],
           [3.5547, 3.5547],
           [3.4966, 3.4966],
           [3.5401, 3.5401],
           [3.5505, 3.5505],
           [3.4872, 3.4872],
           [3.5469, 3.5469],
           [3.5524, 3.5524],
           [3.5577, 3.5577],
           [3.5622, 3.5622],
           [3.5661, 3.5661],
           [3.5693, 3.5693],
           [3.5721, 3.5721],
           [3.5746, 3.5746],
           [3.5767, 3.5767]],
 
          [[3.5647, 3.5647],
           [3.4939, 3.4939],
           [3.5547, 3.5547],
           [3.4966, 3.4966],
           [3.5401, 3.5401],
           [3.5505, 3.5505],
           [3.4872, 3.4872],
           [3.5469, 3.5469],
           [3.5524, 3.5524],
           [3.5577, 3.5577],
           [3.5622, 3.5622],
           [3.5661, 3.5661],
           [3.5693, 3.5693],
           [3.5721, 3.5721],
           [3.5746, 3.5746],
           [3.5767, 3.5767]]],
 
 
         [[[3.4838, 3.4838],
      

**Normalization denominator**

In [55]:
k_sum = k.sum(dim=-2).unsqueeze(-1)
k_sum.shape, k_sum

(torch.Size([2, 2, 2, 1]),
 tensor([[[[13.1377],
           [13.1377]],
 
          [[13.1377],
           [13.1377]]],
 
 
         [[[13.1356],
           [13.1356]],
 
          [[13.1356],
           [13.1356]]]], grad_fn=<UnsqueezeBackward0>))

**Denominator**

In [56]:
z = 1.0 / (q @ k_sum + 1e-6)
z.shape, z

(torch.Size([2, 2, 16, 1]),
 tensor([[[[0.0840],
           [0.0827],
           [0.0839],
           [0.0828],
           [0.0836],
           [0.0838],
           [0.0826],
           [0.0837],
           [0.0838],
           [0.0839],
           [0.0840],
           [0.0841],
           [0.0841],
           [0.0842],
           [0.0842],
           [0.0843]],
 
          [[0.0840],
           [0.0827],
           [0.0839],
           [0.0828],
           [0.0836],
           [0.0838],
           [0.0826],
           [0.0837],
           [0.0838],
           [0.0839],
           [0.0840],
           [0.0841],
           [0.0841],
           [0.0842],
           [0.0842],
           [0.0843]]],
 
 
         [[[0.0826],
           [0.0846],
           [0.0841],
           [0.0830],
           [0.0841],
           [0.0826],
           [0.0836],
           [0.0837],
           [0.0838],
           [0.0839],
           [0.0840],
           [0.0841],
           [0.0841],
           [0.0842

**Numerator**

In [57]:
kv = k.transpose(-2, -1) @ v
kv.shape, kv

(torch.Size([2, 2, 2, 2]),
 tensor([[[[46.6102, 46.6102],
           [46.6102, 46.6102]],
 
          [[46.6102, 46.6102],
           [46.6102, 46.6102]]],
 
 
         [[[46.6404, 46.6404],
           [46.6404, 46.6404]],
 
          [[46.6404, 46.6404],
           [46.6404, 46.6404]]]], grad_fn=<UnsafeViewBackward0>))

In [58]:
qkv = q @ kv
qkv.shape, qkv

(torch.Size([2, 2, 16, 2]),
 tensor([[[[42.2163, 42.2163],
           [42.8857, 42.8857],
           [42.3100, 42.3100],
           [42.8607, 42.8607],
           [42.4477, 42.4477],
           [42.3501, 42.3501],
           [42.9499, 42.9499],
           [42.3836, 42.3836],
           [42.3323, 42.3323],
           [42.2820, 42.2820],
           [42.2397, 42.2397],
           [42.2039, 42.2039],
           [42.1732, 42.1732],
           [42.1467, 42.1467],
           [42.1238, 42.1238],
           [42.1038, 42.1038]],
 
          [[42.2163, 42.2163],
           [42.8857, 42.8857],
           [42.3100, 42.3100],
           [42.8607, 42.8607],
           [42.4477, 42.4477],
           [42.3501, 42.3501],
           [42.9499, 42.9499],
           [42.3836, 42.3836],
           [42.3323, 42.3323],
           [42.2820, 42.2820],
           [42.2397, 42.2397],
           [42.2039, 42.2039],
           [42.1732, 42.1732],
           [42.1467, 42.1467],
           [42.1238, 42.1238],
        

**Linear Attention**

In [59]:
y = qkv * z
y.shape, y

(torch.Size([2, 2, 16, 2]),
 tensor([[[[3.5478, 3.5478],
           [3.5478, 3.5478],
           [3.5478, 3.5478],
           [3.5478, 3.5478],
           [3.5478, 3.5478],
           [3.5478, 3.5478],
           [3.5478, 3.5478],
           [3.5478, 3.5478],
           [3.5478, 3.5478],
           [3.5478, 3.5478],
           [3.5478, 3.5478],
           [3.5478, 3.5478],
           [3.5478, 3.5478],
           [3.5478, 3.5478],
           [3.5478, 3.5478],
           [3.5478, 3.5478]],
 
          [[3.5478, 3.5478],
           [3.5478, 3.5478],
           [3.5478, 3.5478],
           [3.5478, 3.5478],
           [3.5478, 3.5478],
           [3.5478, 3.5478],
           [3.5478, 3.5478],
           [3.5478, 3.5478],
           [3.5478, 3.5478],
           [3.5478, 3.5478],
           [3.5478, 3.5478],
           [3.5478, 3.5478],
           [3.5478, 3.5478],
           [3.5478, 3.5478],
           [3.5478, 3.5478],
           [3.5478, 3.5478]]],
 
 
         [[[3.5507, 3.5507],
      

**Collapse heads**

In [60]:
y = y.transpose(1, 2).contiguous().view(B, T_q, C)
y.shape, y

(torch.Size([2, 16, 4]),
 tensor([[[3.5478, 3.5478, 3.5478, 3.5478],
          [3.5478, 3.5478, 3.5478, 3.5478],
          [3.5478, 3.5478, 3.5478, 3.5478],
          [3.5478, 3.5478, 3.5478, 3.5478],
          [3.5478, 3.5478, 3.5478, 3.5478],
          [3.5478, 3.5478, 3.5478, 3.5478],
          [3.5478, 3.5478, 3.5478, 3.5478],
          [3.5478, 3.5478, 3.5478, 3.5478],
          [3.5478, 3.5478, 3.5478, 3.5478],
          [3.5478, 3.5478, 3.5478, 3.5478],
          [3.5478, 3.5478, 3.5478, 3.5478],
          [3.5478, 3.5478, 3.5478, 3.5478],
          [3.5478, 3.5478, 3.5478, 3.5478],
          [3.5478, 3.5478, 3.5478, 3.5478],
          [3.5478, 3.5478, 3.5478, 3.5478],
          [3.5478, 3.5478, 3.5478, 3.5478]],
 
         [[3.5507, 3.5507, 3.5507, 3.5507],
          [3.5507, 3.5507, 3.5507, 3.5507],
          [3.5507, 3.5507, 3.5507, 3.5507],
          [3.5507, 3.5507, 3.5507, 3.5507],
          [3.5507, 3.5507, 3.5507, 3.5507],
          [3.5507, 3.5507, 3.5507, 3.5507],
    

**Gating**

Sigmoid-based attention gating is a major component we add to self-attention that we do not add to cross-attention. This gating uses the Hadamard product of the gate and the attention, which lets the model learn to suppress or pass through attention output per dimension. This allows the model to decide how much of the attention result to actually use for each dimension. We use a learned linear weight and sigmoid to pull gating values between 0 and 1, allowing the model to turn down specific values.

In [61]:
gate = nn.Linear(embed_dim, embed_dim)
vs, d = embed_dim, embed_dim
rows = torch.arange(vs).unsqueeze(1)
cols = torch.full((d,), 1.0).unsqueeze(0)
pattern = 0.1 * (rows + cols)

gate.weight = nn.Parameter(pattern)
gate.weight

Parameter containing:
tensor([[0.1000, 0.1000, 0.1000, 0.1000],
        [0.2000, 0.2000, 0.2000, 0.2000],
        [0.3000, 0.3000, 0.3000, 0.3000],
        [0.4000, 0.4000, 0.4000, 0.4000]], requires_grad=True)

In [62]:
y = torch.sigmoid(gate(x_norm2)) * y
y.shape, y

(torch.Size([2, 16, 4]),
 tensor([[[2.4094, 2.2796, 2.8220, 2.9718],
          [2.4033, 2.2668, 2.8083, 2.9565],
          [2.4085, 2.2778, 2.8201, 2.9697],
          [2.4035, 2.2672, 2.8088, 2.9571],
          [2.4073, 2.2751, 2.8173, 2.9665],
          [2.4082, 2.2770, 2.8193, 2.9688],
          [2.4027, 2.2655, 2.8070, 2.9550],
          [2.4078, 2.2764, 2.8186, 2.9680],
          [2.4083, 2.2774, 2.8197, 2.9692],
          [2.4088, 2.2783, 2.8207, 2.9703],
          [2.4092, 2.2792, 2.8216, 2.9713],
          [2.4095, 2.2798, 2.8223, 2.9721],
          [2.4098, 2.2804, 2.8229, 2.9728],
          [2.4100, 2.2809, 2.8235, 2.9734],
          [2.4102, 2.2814, 2.8239, 2.9739],
          [2.4104, 2.2818, 2.8244, 2.9744]],
 
         [[2.4044, 2.2667, 2.8086, 2.9566],
          [2.4136, 2.2863, 2.8294, 2.9799],
          [2.4113, 2.2815, 2.8244, 2.9743],
          [2.4063, 2.2708, 2.8130, 2.9615],
          [2.4113, 2.2815, 2.8244, 2.9743],
          [2.4045, 2.2670, 2.8089, 2.9570],
    

**Cross-head final projection**

Finally, we will project the gated attention matrix through a final linear layer. This allows the model to learn how to combine information across the different heads.

In [63]:
c_proj2 = nn.Linear(embed_dim, embed_dim)
vs, d = embed_dim, embed_dim
rows = torch.full((vs,), 0.1).unsqueeze(0)
cols = torch.arange(d).unsqueeze(1)
pattern = 1 * (rows + 0.01 * cols)

c_proj2.weight = nn.Parameter(pattern)
c_proj2.weight

Parameter containing:
tensor([[0.1000, 0.1000, 0.1000, 0.1000],
        [0.1100, 0.1100, 0.1100, 0.1100],
        [0.1200, 0.1200, 0.1200, 0.1200],
        [0.1300, 0.1300, 0.1300, 0.1300]], requires_grad=True)

In [64]:
x_attn = c_proj2(y)
x_attn.shape, x_attn

(torch.Size([2, 16, 4]),
 tensor([[[1.2381, 1.4876, 1.4684, 1.6226],
          [1.2333, 1.4823, 1.4627, 1.6163],
          [1.2374, 1.4868, 1.4676, 1.6217],
          [1.2334, 1.4825, 1.4629, 1.6166],
          [1.2364, 1.4857, 1.4664, 1.6204],
          [1.2371, 1.4865, 1.4673, 1.6213],
          [1.2328, 1.4818, 1.4621, 1.6157],
          [1.2369, 1.4863, 1.4670, 1.6210],
          [1.2372, 1.4867, 1.4674, 1.6215],
          [1.2376, 1.4871, 1.4678, 1.6220],
          [1.2379, 1.4874, 1.4682, 1.6224],
          [1.2382, 1.4877, 1.4685, 1.6227],
          [1.2384, 1.4879, 1.4688, 1.6230],
          [1.2386, 1.4881, 1.4690, 1.6232],
          [1.2387, 1.4883, 1.4692, 1.6234],
          [1.2389, 1.4885, 1.4694, 1.6236]],
 
         [[1.2334, 1.4825, 1.4628, 1.6165],
          [1.2407, 1.4905, 1.4716, 1.6260],
          [1.2389, 1.4885, 1.4695, 1.6237],
          [1.2349, 1.4841, 1.4647, 1.6185],
          [1.2389, 1.4885, 1.4694, 1.6237],
          [1.2335, 1.4826, 1.4630, 1.6167],
    

#### Attention - Residual Connection 2

We now have our gated linear self-attention calculated and will use a residual connection to allow gradients to bypass the attention layer. With this, you'll see how much larger the residual connection's impact is on our output compared to our attention.

In [65]:
x = x + x_attn
x.shape, x

(torch.Size([2, 16, 4]),
 tensor([[[14.4529, 15.0257, 16.0283, 20.1268],
          [12.4471, 14.4599, 14.0521, 21.9174],
          [13.3198, 15.1954, 16.5922, 20.1602],
          [13.0044, 14.8295, 13.8081, 22.2146],
          [15.2660, 14.0427, 14.4857, 20.9894],
          [12.5616, 15.7905, 16.1755, 19.8651],
          [13.4656, 13.0234, 14.2885, 22.2187],
          [13.7652, 13.9578, 18.2380, 20.0380],
          [13.5260, 16.5425, 18.6315, 21.0184],
          [14.5264, 17.5429, 19.6319, 22.0189],
          [15.5267, 18.5433, 20.6323, 23.0193],
          [16.5269, 19.5436, 21.6326, 24.0196],
          [17.5272, 20.5438, 22.6329, 25.0199],
          [18.5274, 21.5440, 23.6331, 26.0202],
          [19.5275, 22.5442, 24.6333, 27.0204],
          [20.5277, 23.5444, 25.6335, 28.0206]],
 
         [[13.4715, 13.9605, 14.2592, 22.9564],
          [16.5464, 15.3450, 17.6570, 18.3189],
          [14.0227, 14.8825, 17.0703, 19.7553],
          [13.4089, 13.6315, 16.3601, 22.3300],
          [1

#### RMSNorm 3

We'll do another round of normalization before the SwiGLU MLP. We'll use RMSNorm again.

In [66]:
rms3 = RMSNorm(embed_dim)
rms3.weight

Parameter containing:
tensor([1., 1., 1., 1.], requires_grad=True)

In [67]:
x_norm3 = rms3(x)
x_norm3.shape, x_norm3

(torch.Size([2, 16, 4]),
 tensor([[[0.8729, 0.9075, 0.9680, 1.2155],
          [0.7712, 0.8960, 0.8707, 1.3580],
          [0.8069, 0.9205, 1.0051, 1.2212],
          [0.7939, 0.9054, 0.8430, 1.3562],
          [0.9288, 0.8544, 0.8813, 1.2770],
          [0.7704, 0.9684, 0.9921, 1.2183],
          [0.8316, 0.8043, 0.8824, 1.3722],
          [0.8232, 0.8347, 1.0907, 1.1983],
          [0.7665, 0.9375, 1.0559, 1.1911],
          [0.7795, 0.9414, 1.0535, 1.1816],
          [0.7912, 0.9449, 1.0513, 1.1730],
          [0.8017, 0.9480, 1.0493, 1.1651],
          [0.8112, 0.9508, 1.0475, 1.1579],
          [0.8198, 0.9533, 1.0457, 1.1514],
          [0.8277, 0.9556, 1.0441, 1.1453],
          [0.8349, 0.9576, 1.0426, 1.1397]],
 
         [[0.8099, 0.8393, 0.8573, 1.3801],
          [0.9731, 0.9024, 1.0384, 1.0773],
          [0.8457, 0.8975, 1.0295, 1.1914],
          [0.7971, 0.8103, 0.9726, 1.3274],
          [0.8550, 0.9027, 1.0026, 1.2038],
          [0.8330, 0.8580, 0.8189, 1.3782],
    

#### Attention - SwiGLU

Now that we have our attention calculated, we'll introduce our nonlinearity layers. Traditionally this was an MLP, but we replaced it with a swish-gated linear unit, or SwiGLU. SwiGLU replaces the single linear transform in a standard MLP with a gated pathway, producing a result as follows:

$$\begin{aligned}
\text{SiLU}(Z) &= Z \odot \sigma(Z) \\
\text{gate} &= \text{SiLU}(x W_g^\top) \\
H &= x W_u^\top \\
y &= (\text{gate} \odot H) W_d^\top
\end{aligned}$$

Gating based on the Hadamard product lets the network learn to selectively amplify or suppress features before the final projection, giving it more expressive power per parameter than a standard two-layer MLP with ReLU/GELU at the cost of some extra compute.

*You'll notice that SwiGLU has 3 weight matrices $W_g, W_u, W_d$, instead of the typical 2 we use in MLP. In production code, you might see helper functions that convert MLP ratios to SwiGLU ratios using 2/3 multiples to maintain the number of parameters.*

In [68]:
hidden_dim = 8

**Gate**

We'll start by initializing our gating weight $W_g$ and calculating our gate. The gate will need to scale up to our hidden dimension as it will multiply directly against our weighted input.

In [69]:
wg = nn.Linear(embed_dim, hidden_dim, bias=False)
nn.init.constant_(wg.weight, 0.5)
wg.weight

Parameter containing:
tensor([[0.5000, 0.5000, 0.5000, 0.5000],
        [0.5000, 0.5000, 0.5000, 0.5000],
        [0.5000, 0.5000, 0.5000, 0.5000],
        [0.5000, 0.5000, 0.5000, 0.5000],
        [0.5000, 0.5000, 0.5000, 0.5000],
        [0.5000, 0.5000, 0.5000, 0.5000],
        [0.5000, 0.5000, 0.5000, 0.5000],
        [0.5000, 0.5000, 0.5000, 0.5000]], requires_grad=True)

In [70]:
xwg = wg(x_norm3)
xwg.shape, xwg

(torch.Size([2, 16, 8]),
 tensor([[[1.9819, 1.9819, 1.9819, 1.9819, 1.9819, 1.9819, 1.9819, 1.9819],
          [1.9480, 1.9480, 1.9480, 1.9480, 1.9480, 1.9480, 1.9480, 1.9480],
          [1.9769, 1.9769, 1.9769, 1.9769, 1.9769, 1.9769, 1.9769, 1.9769],
          [1.9493, 1.9493, 1.9493, 1.9493, 1.9493, 1.9493, 1.9493, 1.9493],
          [1.9707, 1.9707, 1.9707, 1.9707, 1.9707, 1.9707, 1.9707, 1.9707],
          [1.9746, 1.9746, 1.9746, 1.9746, 1.9746, 1.9746, 1.9746, 1.9746],
          [1.9453, 1.9453, 1.9453, 1.9453, 1.9453, 1.9453, 1.9453, 1.9453],
          [1.9735, 1.9735, 1.9735, 1.9735, 1.9735, 1.9735, 1.9735, 1.9735],
          [1.9755, 1.9755, 1.9755, 1.9755, 1.9755, 1.9755, 1.9755, 1.9755],
          [1.9780, 1.9780, 1.9780, 1.9780, 1.9780, 1.9780, 1.9780, 1.9780],
          [1.9802, 1.9802, 1.9802, 1.9802, 1.9802, 1.9802, 1.9802, 1.9802],
          [1.9821, 1.9821, 1.9821, 1.9821, 1.9821, 1.9821, 1.9821, 1.9821],
          [1.9837, 1.9837, 1.9837, 1.9837, 1.9837, 1.9837, 1.98

**SiLU Nonlinearity**

SiLU will pull our negative values closer to zero. Values above 1 will remain almost linear. When combined with the learned weights, you can quickly see how this becomes a gate.

In [71]:
xwg = F.silu(xwg)
xwg.shape, xwg

(torch.Size([2, 16, 8]),
 tensor([[[1.7419, 1.7419, 1.7419, 1.7419, 1.7419, 1.7419, 1.7419, 1.7419],
          [1.7049, 1.7049, 1.7049, 1.7049, 1.7049, 1.7049, 1.7049, 1.7049],
          [1.7364, 1.7364, 1.7364, 1.7364, 1.7364, 1.7364, 1.7364, 1.7364],
          [1.7063, 1.7063, 1.7063, 1.7063, 1.7063, 1.7063, 1.7063, 1.7063],
          [1.7297, 1.7297, 1.7297, 1.7297, 1.7297, 1.7297, 1.7297, 1.7297],
          [1.7339, 1.7339, 1.7339, 1.7339, 1.7339, 1.7339, 1.7339, 1.7339],
          [1.7020, 1.7020, 1.7020, 1.7020, 1.7020, 1.7020, 1.7020, 1.7020],
          [1.7327, 1.7327, 1.7327, 1.7327, 1.7327, 1.7327, 1.7327, 1.7327],
          [1.7349, 1.7349, 1.7349, 1.7349, 1.7349, 1.7349, 1.7349, 1.7349],
          [1.7376, 1.7376, 1.7376, 1.7376, 1.7376, 1.7376, 1.7376, 1.7376],
          [1.7400, 1.7400, 1.7400, 1.7400, 1.7400, 1.7400, 1.7400, 1.7400],
          [1.7420, 1.7420, 1.7420, 1.7420, 1.7420, 1.7420, 1.7420, 1.7420],
          [1.7438, 1.7438, 1.7438, 1.7438, 1.7438, 1.7438, 1.74

**Weighted input**

Now we'll need to scale up our input to the hidden dimension. We'll use a weighted layer $W_u$ allowing the model to determine how to use the different channels to create the new dimensions.

In [72]:
wu = nn.Linear(embed_dim, hidden_dim, bias=False)
nn.init.constant_(wu.weight, -0.1)
wu.weight

Parameter containing:
tensor([[-0.1000, -0.1000, -0.1000, -0.1000],
        [-0.1000, -0.1000, -0.1000, -0.1000],
        [-0.1000, -0.1000, -0.1000, -0.1000],
        [-0.1000, -0.1000, -0.1000, -0.1000],
        [-0.1000, -0.1000, -0.1000, -0.1000],
        [-0.1000, -0.1000, -0.1000, -0.1000],
        [-0.1000, -0.1000, -0.1000, -0.1000],
        [-0.1000, -0.1000, -0.1000, -0.1000]], requires_grad=True)

In [73]:
xwu = wu(x_norm3)
xwu.shape, xwu

(torch.Size([2, 16, 8]),
 tensor([[[-0.3964, -0.3964, -0.3964, -0.3964, -0.3964, -0.3964, -0.3964,
           -0.3964],
          [-0.3896, -0.3896, -0.3896, -0.3896, -0.3896, -0.3896, -0.3896,
           -0.3896],
          [-0.3954, -0.3954, -0.3954, -0.3954, -0.3954, -0.3954, -0.3954,
           -0.3954],
          [-0.3899, -0.3899, -0.3899, -0.3899, -0.3899, -0.3899, -0.3899,
           -0.3899],
          [-0.3941, -0.3941, -0.3941, -0.3941, -0.3941, -0.3941, -0.3941,
           -0.3941],
          [-0.3949, -0.3949, -0.3949, -0.3949, -0.3949, -0.3949, -0.3949,
           -0.3949],
          [-0.3891, -0.3891, -0.3891, -0.3891, -0.3891, -0.3891, -0.3891,
           -0.3891],
          [-0.3947, -0.3947, -0.3947, -0.3947, -0.3947, -0.3947, -0.3947,
           -0.3947],
          [-0.3951, -0.3951, -0.3951, -0.3951, -0.3951, -0.3951, -0.3951,
           -0.3951],
          [-0.3956, -0.3956, -0.3956, -0.3956, -0.3956, -0.3956, -0.3956,
           -0.3956],
          [-0.3960, -0.39

**Apply Gate**

Now we'll go ahead and apply the gate. We take the Hadamard product, which allows the model to gate each value of the scaled-up projection.

In [74]:
xw = xwg * xwu
xw.shape, xw

(torch.Size([2, 16, 8]),
 tensor([[[-0.6905, -0.6905, -0.6905, -0.6905, -0.6905, -0.6905, -0.6905,
           -0.6905],
          [-0.6642, -0.6642, -0.6642, -0.6642, -0.6642, -0.6642, -0.6642,
           -0.6642],
          [-0.6865, -0.6865, -0.6865, -0.6865, -0.6865, -0.6865, -0.6865,
           -0.6865],
          [-0.6652, -0.6652, -0.6652, -0.6652, -0.6652, -0.6652, -0.6652,
           -0.6652],
          [-0.6817, -0.6817, -0.6817, -0.6817, -0.6817, -0.6817, -0.6817,
           -0.6817],
          [-0.6848, -0.6848, -0.6848, -0.6848, -0.6848, -0.6848, -0.6848,
           -0.6848],
          [-0.6621, -0.6621, -0.6621, -0.6621, -0.6621, -0.6621, -0.6621,
           -0.6621],
          [-0.6839, -0.6839, -0.6839, -0.6839, -0.6839, -0.6839, -0.6839,
           -0.6839],
          [-0.6854, -0.6854, -0.6854, -0.6854, -0.6854, -0.6854, -0.6854,
           -0.6854],
          [-0.6874, -0.6874, -0.6874, -0.6874, -0.6874, -0.6874, -0.6874,
           -0.6874],
          [-0.6891, -0.68

**Project Down**

Now we need to project back down to our embedding dimension. We'll use a final weighted $W_d$ layer to determine how to project back down. This is similar to the final layer of an MLP.

In [75]:
wd = nn.Linear(hidden_dim, embed_dim, bias=False)
nn.init.constant_(wd.weight, 0.33)
wd.weight

Parameter containing:
tensor([[0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300],
        [0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300],
        [0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300],
        [0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300, 0.3300]],
       requires_grad=True)

In [76]:
xswig = wd(xw)
xswig.shape, xswig

(torch.Size([2, 16, 4]),
 tensor([[[-1.8229, -1.8229, -1.8229, -1.8229],
          [-1.7536, -1.7536, -1.7536, -1.7536],
          [-1.8124, -1.8124, -1.8124, -1.8124],
          [-1.7562, -1.7562, -1.7562, -1.7562],
          [-1.7998, -1.7998, -1.7998, -1.7998],
          [-1.8078, -1.8078, -1.8078, -1.8078],
          [-1.7481, -1.7481, -1.7481, -1.7481],
          [-1.8054, -1.8054, -1.8054, -1.8054],
          [-1.8096, -1.8096, -1.8096, -1.8096],
          [-1.8148, -1.8148, -1.8148, -1.8148],
          [-1.8193, -1.8193, -1.8193, -1.8193],
          [-1.8231, -1.8231, -1.8231, -1.8231],
          [-1.8264, -1.8264, -1.8264, -1.8264],
          [-1.8293, -1.8293, -1.8293, -1.8293],
          [-1.8319, -1.8319, -1.8319, -1.8319],
          [-1.8341, -1.8341, -1.8341, -1.8341]],
 
         [[-1.7441, -1.7441, -1.7441, -1.7441],
          [-1.8510, -1.8510, -1.8510, -1.8510],
          [-1.8231, -1.8231, -1.8231, -1.8231],
          [-1.7652, -1.7652, -1.7652, -1.7652],
          [-

#### Residual Connection 3

We now have our SwiGLU-based projection calculated and will use a residual connection to allow gradients to bypass the SwiGLU calculations. With this, you'll see how much larger the residual connection's impact is on our output compared to our SwiGLU output.

In [77]:
x = x + xswig
x.shape, x

(torch.Size([2, 16, 4]),
 tensor([[[12.6300, 13.2028, 14.2055, 18.3039],
          [10.6935, 12.7063, 12.2986, 20.1638],
          [11.5074, 13.3831, 14.7798, 18.3478],
          [11.2482, 13.0733, 12.0519, 20.4584],
          [13.4662, 12.2429, 12.6859, 19.1896],
          [10.7538, 13.9827, 14.3677, 18.0573],
          [11.7175, 11.2753, 12.5404, 20.4707],
          [11.9598, 12.1523, 16.4326, 18.2326],
          [11.7165, 14.7330, 16.8219, 19.2089],
          [12.7116, 15.7282, 17.8172, 20.2041],
          [13.7074, 16.7240, 18.8130, 21.2000],
          [14.7038, 17.7205, 19.8095, 22.1965],
          [15.7007, 18.7174, 20.8064, 23.1935],
          [16.6980, 19.7147, 21.8038, 24.1908],
          [17.6956, 20.7123, 22.8014, 25.1885],
          [18.6935, 21.7102, 23.7994, 26.1864]],
 
         [[11.7274, 12.2164, 12.5151, 21.2123],
          [14.6954, 13.4939, 15.8059, 16.4678],
          [12.1997, 13.0595, 15.2472, 17.9322],
          [11.6437, 11.8663, 14.5950, 20.5649],
          [1

### Final Layer Normalization

The previous layers can run sequentially for as many transformer layers as configured. The more layers, the "deeper" the network becomes and the more parameters/capacity the model has (not always good). Once all the layers have completed, we're ready for a final normalization. Like previous normalizations, this will pull our values together to focus on the variance. This normalized output is what we'll use to extract our predictions.

In [78]:
rmsf = RMSNorm(embed_dim)
rmsf.weight

Parameter containing:
tensor([1., 1., 1., 1.], requires_grad=True)

In [79]:
sequence = rmsf(x)
sequence.shape, sequence

(torch.Size([2, 16, 4]),
 tensor([[[0.8561, 0.8949, 0.9629, 1.2407],
          [0.7407, 0.8802, 0.8519, 1.3967],
          [0.7818, 0.9092, 1.0041, 1.2465],
          [0.7666, 0.8910, 0.8213, 1.3943],
          [0.9182, 0.8348, 0.8650, 1.3084],
          [0.7405, 0.9628, 0.9893, 1.2434],
          [0.8082, 0.7777, 0.8650, 1.4120],
          [0.8004, 0.8132, 1.0997, 1.2201],
          [0.7387, 0.9289, 1.0606, 1.2111],
          [0.7548, 0.9339, 1.0579, 1.1996],
          [0.7690, 0.9382, 1.0554, 1.1893],
          [0.7817, 0.9421, 1.0531, 1.1800],
          [0.7931, 0.9455, 1.0510, 1.1716],
          [0.8034, 0.9485, 1.0490, 1.1639],
          [0.8127, 0.9512, 1.0472, 1.1568],
          [0.8212, 0.9537, 1.0455, 1.1503]],
 
         [[0.7847, 0.8174, 0.8374, 1.4194],
          [0.9695, 0.8902, 1.0427, 1.0864],
          [0.8256, 0.8838, 1.0318, 1.2135],
          [0.7710, 0.7857, 0.9664, 1.3617],
          [0.8359, 0.8894, 1.0016, 1.2276],
          [0.8110, 0.8386, 0.7952, 1.4166],
    

### Extract Predicted Gene Representations

Now we extract only the query positions from the sequence. Recall that we concatenated `[context_latents, queries]` to create our sequence `x`. Since the context latents come first, everything after the context length contains the predicted representations for our target gene queries.

In [80]:
predictions = sequence[:, num_genes:, :]
predictions.shape, predictions

(torch.Size([2, 8, 4]),
 tensor([[[0.7387, 0.9289, 1.0606, 1.2111],
          [0.7548, 0.9339, 1.0579, 1.1996],
          [0.7690, 0.9382, 1.0554, 1.1893],
          [0.7817, 0.9421, 1.0531, 1.1800],
          [0.7931, 0.9455, 1.0510, 1.1716],
          [0.8034, 0.9485, 1.0490, 1.1639],
          [0.8127, 0.9512, 1.0472, 1.1568],
          [0.8212, 0.9537, 1.0455, 1.1503]],
 
         [[0.7387, 0.9289, 1.0606, 1.2111],
          [0.7548, 0.9339, 1.0579, 1.1996],
          [0.7690, 0.9382, 1.0554, 1.1893],
          [0.7817, 0.9421, 1.0531, 1.1800],
          [0.7931, 0.9455, 1.0510, 1.1716],
          [0.8034, 0.9485, 1.0490, 1.1638],
          [0.8127, 0.9512, 1.0472, 1.1568],
          [0.8212, 0.9537, 1.0455, 1.1503]]], grad_fn=<SliceBackward0>))

### Mean Prediction Head $\mu$

The mean head projects each gene's predicted representation from the predictor embedding dimensions back into the Cell State Encoder dimensions. This gives us the average (mean) predicted latent representation for each gene after the perturbation.

In [81]:
head_mu = nn.Linear(embed_dim, output_dim)
vs, d = embed_dim, output_dim
rows = torch.arange(vs).unsqueeze(0)
cols = torch.arange(d).unsqueeze(1)
pattern = 1.0 * (rows + 1 * cols)
head_mu.weight = nn.Parameter(pattern)
nn.init.zeros_(head_mu.bias)
head_mu.weight, head_mu.bias

(Parameter containing:
 tensor([[0., 1., 2., 3.],
         [1., 2., 3., 4.],
         [2., 3., 4., 5.],
         [3., 4., 5., 6.],
         [4., 5., 6., 7.],
         [5., 6., 7., 8.]], requires_grad=True),
 Parameter containing:
 tensor([0., 0., 0., 0., 0., 0.], requires_grad=True))

In [82]:
mu = head_mu(predictions)
mu.shape, mu

(torch.Size([2, 8, 6]),
 tensor([[[ 6.6832, 10.6225, 14.5617, 18.5009, 22.4401, 26.3793],
          [ 6.6485, 10.5947, 14.5408, 18.4869, 22.4331, 26.3792],
          [ 6.6170, 10.5690, 14.5209, 18.4729, 22.4248, 26.3768],
          [ 6.5883, 10.5452, 14.5021, 18.4590, 22.4158, 26.3727],
          [ 6.5621, 10.5232, 14.4843, 18.4454, 22.4065, 26.3676],
          [ 6.5381, 10.5028, 14.4675, 18.4322, 22.3969, 26.3616],
          [ 6.5159, 10.4838, 14.4517, 18.4195, 22.3874, 26.3552],
          [ 6.4955, 10.4661, 14.4367, 18.4073, 22.3779, 26.3485]],
 
         [[ 6.6832, 10.6225, 14.5617, 18.5009, 22.4401, 26.3793],
          [ 6.6485, 10.5947, 14.5408, 18.4869, 22.4331, 26.3792],
          [ 6.6170, 10.5690, 14.5209, 18.4729, 22.4248, 26.3768],
          [ 6.5883, 10.5452, 14.5021, 18.4590, 22.4158, 26.3727],
          [ 6.5621, 10.5232, 14.4843, 18.4454, 22.4065, 26.3676],
          [ 6.5381, 10.5028, 14.4675, 18.4322, 22.3969, 26.3616],
          [ 6.5159, 10.4838, 14.4517, 18.4195, 22

### Log-Variance Prediction Head $\log\sigma^2$

The log-variance head projects each gene's prediction into an uncertainty estimate for each dimension in the Cell State Encoder space. Higher values indicate the model is less confident in that dimension of its prediction. We predict log-variance rather than variance directly because the log scale is numerically stable, unconstrained and closer together, so there's less incentive to get only large values correct.

In [83]:
head_logvar = nn.Linear(embed_dim, output_dim)
vs, d = embed_dim, output_dim
rows = torch.arange(vs).unsqueeze(0)
cols = torch.arange(d).unsqueeze(1)
pattern = 0.01 * (rows - 0.1 * cols)
head_logvar.weight = nn.Parameter(pattern)
nn.init.zeros_(head_logvar.bias)
head_logvar.weight, head_logvar.bias

(Parameter containing:
 tensor([[ 0.0000,  0.0100,  0.0200,  0.0300],
         [-0.0010,  0.0090,  0.0190,  0.0290],
         [-0.0020,  0.0080,  0.0180,  0.0280],
         [-0.0030,  0.0070,  0.0170,  0.0270],
         [-0.0040,  0.0060,  0.0160,  0.0260],
         [-0.0050,  0.0050,  0.0150,  0.0250]], requires_grad=True),
 Parameter containing:
 tensor([0., 0., 0., 0., 0., 0.], requires_grad=True))

In [84]:
logvar = head_logvar(predictions.float())
logvar.shape, logvar

(torch.Size([2, 8, 6]),
 tensor([[[0.0668, 0.0629, 0.0590, 0.0550, 0.0511, 0.0471],
          [0.0665, 0.0625, 0.0586, 0.0546, 0.0507, 0.0468],
          [0.0662, 0.0622, 0.0583, 0.0543, 0.0504, 0.0464],
          [0.0659, 0.0619, 0.0580, 0.0540, 0.0501, 0.0461],
          [0.0656, 0.0617, 0.0577, 0.0537, 0.0498, 0.0458],
          [0.0654, 0.0614, 0.0575, 0.0535, 0.0495, 0.0456],
          [0.0652, 0.0612, 0.0572, 0.0533, 0.0493, 0.0453],
          [0.0650, 0.0610, 0.0570, 0.0530, 0.0491, 0.0451]],
 
         [[0.0668, 0.0629, 0.0590, 0.0550, 0.0511, 0.0471],
          [0.0665, 0.0625, 0.0586, 0.0546, 0.0507, 0.0468],
          [0.0662, 0.0622, 0.0583, 0.0543, 0.0504, 0.0464],
          [0.0659, 0.0619, 0.0580, 0.0540, 0.0501, 0.0461],
          [0.0656, 0.0617, 0.0577, 0.0537, 0.0498, 0.0458],
          [0.0654, 0.0614, 0.0575, 0.0535, 0.0495, 0.0456],
          [0.0652, 0.0612, 0.0572, 0.0533, 0.0493, 0.0453],
          [0.0650, 0.0610, 0.0570, 0.0530, 0.0491, 0.0451]]],
        gra

**Clamp logvar**

We bound the predicted variance to a sane range. Since $\text{var} = e^{\text{logvar}}$:

- $\text{logvar} = -10 \implies \text{var} \approx 0.00005$ (very confident, near-deterministic)
- $\text{logvar} = 2 \implies \text{var} \approx 7.4$ (high uncertainty)

Without clamping, the model could predict extreme log-variance values early in training. Very negative values produce near-zero variance, causing the prediction error term in Gaussian NLL to grow quickly. Very positive values reduce the influence of the prediction error, although Gaussian NLL still penalizes the larger uncertainty. The clamp keeps both behaviors within a controlled range.

In [85]:
logvar = torch.clamp(logvar, min=-10, max=2)
logvar.shape, logvar

(torch.Size([2, 8, 6]),
 tensor([[[0.0668, 0.0629, 0.0590, 0.0550, 0.0511, 0.0471],
          [0.0665, 0.0625, 0.0586, 0.0546, 0.0507, 0.0468],
          [0.0662, 0.0622, 0.0583, 0.0543, 0.0504, 0.0464],
          [0.0659, 0.0619, 0.0580, 0.0540, 0.0501, 0.0461],
          [0.0656, 0.0617, 0.0577, 0.0537, 0.0498, 0.0458],
          [0.0654, 0.0614, 0.0575, 0.0535, 0.0495, 0.0456],
          [0.0652, 0.0612, 0.0572, 0.0533, 0.0493, 0.0453],
          [0.0650, 0.0610, 0.0570, 0.0530, 0.0491, 0.0451]],
 
         [[0.0668, 0.0629, 0.0590, 0.0550, 0.0511, 0.0471],
          [0.0665, 0.0625, 0.0586, 0.0546, 0.0507, 0.0468],
          [0.0662, 0.0622, 0.0583, 0.0543, 0.0504, 0.0464],
          [0.0659, 0.0619, 0.0580, 0.0540, 0.0501, 0.0461],
          [0.0656, 0.0617, 0.0577, 0.0537, 0.0498, 0.0458],
          [0.0654, 0.0614, 0.0575, 0.0535, 0.0495, 0.0456],
          [0.0652, 0.0612, 0.0572, 0.0533, 0.0493, 0.0453],
          [0.0650, 0.0610, 0.0570, 0.0530, 0.0491, 0.0451]]],
        gra

## AC Predictor Complete

For the perturbed cell, we now have mean ($\mu$) and log-variance ($\log\sigma^2$) predictions for each gene and latent dimension. You'll see that both outputs have shape $[B, \text{num\_genes}, \text{output\_dim}]$, one latent vector per gene per sample.

In [86]:
mu.shape, mu

(torch.Size([2, 8, 6]),
 tensor([[[ 6.6832, 10.6225, 14.5617, 18.5009, 22.4401, 26.3793],
          [ 6.6485, 10.5947, 14.5408, 18.4869, 22.4331, 26.3792],
          [ 6.6170, 10.5690, 14.5209, 18.4729, 22.4248, 26.3768],
          [ 6.5883, 10.5452, 14.5021, 18.4590, 22.4158, 26.3727],
          [ 6.5621, 10.5232, 14.4843, 18.4454, 22.4065, 26.3676],
          [ 6.5381, 10.5028, 14.4675, 18.4322, 22.3969, 26.3616],
          [ 6.5159, 10.4838, 14.4517, 18.4195, 22.3874, 26.3552],
          [ 6.4955, 10.4661, 14.4367, 18.4073, 22.3779, 26.3485]],
 
         [[ 6.6832, 10.6225, 14.5617, 18.5009, 22.4401, 26.3793],
          [ 6.6485, 10.5947, 14.5408, 18.4869, 22.4331, 26.3792],
          [ 6.6170, 10.5690, 14.5209, 18.4729, 22.4248, 26.3768],
          [ 6.5883, 10.5452, 14.5021, 18.4590, 22.4158, 26.3727],
          [ 6.5621, 10.5232, 14.4843, 18.4454, 22.4065, 26.3676],
          [ 6.5381, 10.5028, 14.4675, 18.4322, 22.3969, 26.3616],
          [ 6.5159, 10.4838, 14.4517, 18.4195, 22

In [87]:
logvar.shape, logvar

(torch.Size([2, 8, 6]),
 tensor([[[0.0668, 0.0629, 0.0590, 0.0550, 0.0511, 0.0471],
          [0.0665, 0.0625, 0.0586, 0.0546, 0.0507, 0.0468],
          [0.0662, 0.0622, 0.0583, 0.0543, 0.0504, 0.0464],
          [0.0659, 0.0619, 0.0580, 0.0540, 0.0501, 0.0461],
          [0.0656, 0.0617, 0.0577, 0.0537, 0.0498, 0.0458],
          [0.0654, 0.0614, 0.0575, 0.0535, 0.0495, 0.0456],
          [0.0652, 0.0612, 0.0572, 0.0533, 0.0493, 0.0453],
          [0.0650, 0.0610, 0.0570, 0.0530, 0.0491, 0.0451]],
 
         [[0.0668, 0.0629, 0.0590, 0.0550, 0.0511, 0.0471],
          [0.0665, 0.0625, 0.0586, 0.0546, 0.0507, 0.0468],
          [0.0662, 0.0622, 0.0583, 0.0543, 0.0504, 0.0464],
          [0.0659, 0.0619, 0.0580, 0.0540, 0.0501, 0.0461],
          [0.0656, 0.0617, 0.0577, 0.0537, 0.0498, 0.0458],
          [0.0654, 0.0614, 0.0575, 0.0535, 0.0495, 0.0456],
          [0.0652, 0.0612, 0.0572, 0.0533, 0.0493, 0.0453],
          [0.0650, 0.0610, 0.0570, 0.0530, 0.0491, 0.0451]]],
        gra

# Latent Representation

We now have a latent representation of our predicted perturbed cell states. You'll notice that the shape still contains the batch, our context length (number of genes), and the output dimensions, which match the Cell State Encoder dimensions. The ACPredictor works within this latent space to predict a mean latent representation and an uncertainty estimate. During training, the predicted mean and log-variance are used for the Gaussian NLL loss, while VICReg is calculated using only the predicted mean and the teacher encoder's target representation.